# Paper Summary Tables

The selected judges are GPT-5.4 mini, Qwen 3.6 35B, and Gemma 4 31B. Llama is excluded. Failed or missing ratings are excluded from means, sample SDs, and agreement, and valid judge counts are reported explicitly.

LLM metrics cover all three question types. Final and total recall cover only entity questions, excluding empty annotations. Overall means pool valid item-level scores within each judge before averaging judge means with equal weight.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "evaluation_results.py").exists())
BASE_DIR = REPO_ROOT / "evaluations" / "chatbs-base"
ANSWER_REPORT = "original"
WINRATE_DIRNAME = f"answer_winrate-{ANSWER_REPORT}"
JUDGE_IDS = ["gpt-5.4-mini", "qwen3.6-35b-a3b", "gemma-4-31b"]
JUDGE_LABELS = {
    "gemma-4-31b": "Gemma 4 31B",
    "gpt-5.4-mini": "GPT-5.4 mini",
    "qwen3.6-35b-a3b": "Qwen 3.6 35B",
}

TABLE_OUTPUT_DIR = BASE_DIR / "paper_tables" / "judges" / "ensemble"
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def read_csv_or_empty(path):
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def judge_analysis_dir(judge_id):
    return BASE_DIR / "analysis" / "judges" / judge_id

BASE_DIR

PosixPath('/home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base')

## Load and Aggregate Judge Results

Each `*_mean` value is the equal-weight mean of the available judge-level means. Each corresponding `*_std` is the sample standard deviation across those judge-level means, not the within-question standard deviation stored in an individual judge's source summary. Per-judge source rows are retained in `judge_sources` and exported below.

In [2]:
LLM_METRICS = ["llm_completeness","llm_relevance","llm_faithfulness","llm_understanderbility"]
CATEGORY_NAMES = ["bool", "entity", "numeric"]
availability_rows = []
per_judge_summaries = {category: [] for category in CATEGORY_NAMES}
winrate_frames = []
pairwise_frames = []

for judge_id in JUDGE_IDS:
    analysis_dir = judge_analysis_dir(judge_id)
    for category in CATEGORY_NAMES:
        path = analysis_dir / category / "results_summary.csv"
        frame = read_csv_or_empty(path)
        raw = read_csv_or_empty(analysis_dir / category / "results.csv")
        llm_values = raw.reindex(columns=LLM_METRICS).apply(pd.to_numeric, errors="coerce")
        availability_rows.append({
            "judge_id": judge_id,
            "judge": JUDGE_LABELS[judge_id],
            "artifact": f"{category}_summary",
            "available": not frame.empty,
            "valid_llm_rows": int(llm_values.notna().all(axis=1).sum()),
            "llm_error_rows": int(raw.get("llm_answer_quality_error", pd.Series(dtype=str)).notna().sum()),
            "path": str(path.relative_to(REPO_ROOT)),
        })
        if not frame.empty:
            if category != "entity":
                frame = frame.drop(columns=[column for column in frame if column.startswith("entity_")])
            elif not raw.empty and "entity_recall_final" in raw:
                frame["entity_recall_final_count"] = frame["run"].map(
                    raw.groupby("run")["entity_recall_final"].count()
                )
            frame.insert(0, "judge_id", judge_id)
            frame.insert(1, "judge", JUDGE_LABELS[judge_id])
            per_judge_summaries[category].append(frame)

    winrate_path = analysis_dir / WINRATE_DIRNAME / "answer_winrate_summary.csv"
    winrate = read_csv_or_empty(winrate_path)
    availability_rows.append({
        "judge_id": judge_id,
        "judge": JUDGE_LABELS[judge_id],
        "artifact": "answer_winrate_summary",
        "available": not winrate.empty,
        "path": str(winrate_path.relative_to(REPO_ROOT)),
    })
    if not winrate.empty:
        winrate.insert(0, "judge_id", judge_id)
        winrate.insert(1, "judge", JUDGE_LABELS[judge_id])
        winrate_frames.append(winrate)

    pairwise_path = analysis_dir / WINRATE_DIRNAME / "pairwise_answer_winrate.csv"
    pairwise = read_csv_or_empty(pairwise_path)
    availability_rows.append({
        "judge_id": judge_id,
        "judge": JUDGE_LABELS[judge_id],
        "artifact": "pairwise_answer_winrate",
        "available": not pairwise.empty,
        "path": str(pairwise_path.relative_to(REPO_ROOT)),
    })
    if not pairwise.empty:
        pairwise["judge_id"] = judge_id
        pairwise["judge"] = JUDGE_LABELS[judge_id]
        pairwise_frames.append(pairwise)

availability = pd.DataFrame(availability_rows)


def aggregate_judge_values(source, group_column, metric_columns):
    rows = []
    for group_name, group in source.groupby(group_column, dropna=False):
        counts = pd.to_numeric(group["evaluated_examples"], errors="coerce")
        row = {
            group_column: group_name,
            "judge_count": group["judge_id"].nunique(),
            "evaluated_examples": counts.min(),
            "evaluated_examples_min": counts.min(),
            "evaluated_examples_max": counts.max(),
        }
        row["llm_judge_count"] = int(group.reindex(
            columns=[f"{metric}_mean" for metric in LLM_METRICS]
        ).notna().any(axis=1).sum())
        if "entity_recall_final_count" in group:
            row["recall_examples"] = pd.to_numeric(group["entity_recall_final_count"], errors="coerce").min()
        for column in metric_columns:
            values = pd.to_numeric(group[column], errors="coerce")
            row[column] = values.mean()
            row[column.removesuffix("_mean") + "_judge_count"] = int(values.notna().sum())
            row[column.removesuffix("_mean") + "_std"] = values.std(ddof=1)
        rows.append(row)
    return pd.DataFrame(rows)


judge_sources = {}
summaries = {}
for category in CATEGORY_NAMES:
    if not per_judge_summaries[category]:
        continue
    source = pd.concat(per_judge_summaries[category], ignore_index=True)
    metric_columns = [column for column in source.columns if column.endswith("_mean")]
    judge_sources[category] = source
    summaries[category] = aggregate_judge_values(source, "run", metric_columns)

if winrate_frames:
    winrate_source = pd.concat(winrate_frames, ignore_index=True)
    winrate_rows = []
    for method, group in winrate_source.groupby("method", dropna=False):
        row = {
            "method": method,
            "judge_count": group["judge_id"].nunique(),
            "comparisons_min": group["comparisons"].min(),
            "comparisons_max": group["comparisons"].max(),
        }
        for column in ["wins", "losses", "ties", "winrate"]:
            values = pd.to_numeric(group[column], errors="coerce")
            row[f"{column}_mean"] = values.mean()
            row[f"{column}_std"] = values.std(ddof=1)
        winrate_rows.append(row)
    summaries["answer_winrate"] = pd.DataFrame(winrate_rows)
else:
    winrate_source = pd.DataFrame()

if pairwise_frames:
    summaries["pairwise_answer_winrate"] = pd.concat(pairwise_frames, ignore_index=True)

display(availability)
summaries.keys()

,judge_id,judge,artifact,available,valid_llm_rows,llm_error_rows,path
0,gpt-5.4-mini,GPT-5.4 mini,bool_summary,True,266.0,0.0,evaluations/chatbs-base/analysis/judges/gpt-5....
1,gpt-5.4-mini,GPT-5.4 mini,entity_summary,True,343.0,0.0,evaluations/chatbs-base/analysis/judges/gpt-5....
2,gpt-5.4-mini,GPT-5.4 mini,numeric_summary,True,112.0,0.0,evaluations/chatbs-base/analysis/judges/gpt-5....
3,gpt-5.4-mini,GPT-5.4 mini,answer_winrate_summary,True,NaN,NaN,evaluations/chatbs-base/analysis/judges/gpt-5....
4,gpt-5.4-mini,GPT-5.4 mini,pairwise_answer_winrate,True,NaN,NaN,evaluations/chatbs-base/analysis/judges/gpt-5....
5,qwen3.6-35b-a3b,Qwen 3.6 35B,bool_summary,True,262.0,4.0,evaluations/chatbs-base/analysis/judges/qwen3....
6,qwen3.6-35b-a3b,Qwen 3.6 35B,entity_summary,True,338.0,5.0,evaluations/chatbs-base/analysis/judges/qwen3....
7,qwen3.6-35b-a3b,Qwen 3.6 35B,numeric_summary,True,112.0,0.0,evaluations/chatbs-base/analysis/judges/qwen3....
8,qwen3.6-35b-a3b,Qwen 3.6 35B,answer_winrate_summary,True,NaN,NaN,evaluations/chatbs-base/analysis/judges/qwen3....
9,qwen3.6-35b-a3b,Qwen 3.6 35B,pairwise_answer_winrate,True,NaN,NaN,evaluations/chatbs-base/analysis/judges/qwen3....


dict_keys(['bool', 'entity', 'numeric', 'answer_winrate', 'pairwise_answer_winrate'])

In [3]:
# Quick preview of each loaded summary.
for name, df in summaries.items():
    print(f"\n{name}: {df.shape[0]} rows x {df.shape[1]} columns")
    display(df.head())


bool: 7 rows x 51 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,llm_judge_count,answer_token_precision_mean,answer_token_precision_judge_count,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_judge_count,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_judge_count,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_judge_count,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_judge_count,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_judge_count,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_judge_count,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_judge_count,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_judge_count,bertscore_f1_std,llm_completeness_mean,llm_completeness_judge_count,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_judge_count,llm_faithfulness_std,llm_relevance_mean,llm_relevance_judge_count,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_judge_count,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_judge_count,nli_entailment_max_std,bool_accuracy_mean,bool_accuracy_judge_count,bool_accuracy_std
0,fullcontext,3,38,38,38,3,0.637632,3,0.0,0.472221,3,0.0,0.496864,3,0.000000e+00,2.289474,3,0.0,0.394737,3,0.0,0.146592,3,0.000000e+00,0.808490,3,0.000000e+00,0.836027,3,0.000000e+00,0.821554,3,0.000000e+00,0.842982,3,0.030501,0.835088,3,0.015698,0.984035,3,0.021186,0.981228,3,0.019807,0.655778,3,0.0,0.894737,3,0.0
1,grasp,3,38,38,38,3,0.106623,3,0.0,0.093875,3,0.0,0.085124,3,1.699675e-17,2.289474,3,0.0,0.026316,3,0.0,0.002024,3,0.000000e+00,0.840972,3,0.000000e+00,0.766515,3,1.359740e-16,0.800113,3,1.359740e-16,0.263158,3,0.055138,0.278070,3,0.067058,0.479825,3,0.077813,0.713860,3,0.205594,0.492698,3,0.0,0.289474,3,0.0
2,hipporag,3,38,38,38,3,0.000000,3,0.0,0.000000,3,0.0,0.000000,3,0.000000e+00,2.289474,3,0.0,0.289474,3,0.0,0.206140,3,3.399350e-17,0.709924,3,0.000000e+00,0.762432,3,0.000000e+00,0.735114,3,0.000000e+00,0.391003,3,0.107671,0.452857,3,0.064064,0.958195,3,0.044494,0.998008,3,0.003226,0.260601,3,0.0,0.526316,3,0.0
3,hypergraphrag,3,38,38,38,3,0.236746,3,0.0,0.418532,3,0.0,0.258888,3,0.000000e+00,2.289474,3,0.0,0.052632,3,0.0,0.008603,3,0.000000e+00,0.833537,3,1.359740e-16,0.806944,3,0.000000e+00,0.819021,3,0.000000e+00,0.387281,3,0.010716,0.523421,3,0.165219,0.553158,3,0.022348,0.969825,3,0.029025,0.503595,3,0.0,0.421053,3,0.0
4,llmbased,3,38,38,38,3,0.347558,3,0.0,0.165251,3,0.0,0.205004,3,0.000000e+00,2.289474,3,0.0,0.000000,3,0.0,0.000000,3,0.000000e+00,0.781740,3,0.000000e+00,0.776166,3,1.359740e-16,0.778511,3,0.000000e+00,0.100296,3,0.032878,0.121302,3,0.037457,0.200830,3,0.093134,0.717568,3,0.099693,0.277763,3,0.0,0.000000,3,0.0



entity: 7 rows x 76 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,llm_judge_count,recall_examples,answer_token_precision_mean,answer_token_precision_judge_count,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_judge_count,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_judge_count,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_judge_count,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_judge_count,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_judge_count,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_judge_count,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_judge_count,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_judge_count,bertscore_f1_std,llm_completeness_mean,llm_completeness_judge_count,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_judge_count,llm_faithfulness_std,llm_relevance_mean,llm_relevance_judge_count,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_judge_count,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_judge_count,nli_entailment_max_std,entity_gt_total_mean,entity_gt_total_judge_count,entity_gt_total_std,entity_retrieved_final_total_mean,entity_retrieved_final_total_judge_count,entity_retrieved_final_total_std,entity_retrieved_total_total_mean,entity_retrieved_total_total_judge_count,entity_retrieved_total_total_std,entity_recall_final_mean,entity_recall_final_judge_count,entity_recall_final_std,entity_precision_final_mean,entity_precision_final_judge_count,entity_precision_final_std,entity_f1_final_mean,entity_f1_final_judge_count,entity_f1_final_std,entity_recall_total_mean,entity_recall_total_judge_count,entity_recall_total_std,entity_precision_total_mean,entity_precision_total_judge_count,entity_precision_total_std,entity_f1_total_mean,entity_f1_total_judge_count,entity_f1_total_std
0,fullcontext,3,49,49,49,3,43,0.410651,3,0.0,0.285590,3,0.0,0.250826,3,0.0,2.673469,3,0.0,0.428571,3,6.798700e-17,0.350775,3,0.0,0.834169,3,0.0,0.843230,3,1.359740e-16,0.838148,3,1.359740e-16,0.669255,3,0.057569,0.694008,3,0.058629,0.864345,3,0.078766,0.929762,3,0.052470,0.314016,3,0.0,2.55102,3,0.0,6.857143,3,0.0,11.142857,3,2.175584e-15,0.566949,3,0.000000e+00,0.160490,3,0.0,0.379824,3,0.0,0.658386,3,0.0,0.112145,3,1.699675e-17,0.268076,3,0.0
1,grasp,3,49,49,49,3,43,0.185071,3,0.0,0.112896,3,0.0,0.097409,3,0.0,2.673469,3,0.0,0.020408,3,0.000000e+00,0.001938,3,0.0,0.826952,3,0.0,0.786332,3,0.000000e+00,0.805476,3,0.000000e+00,0.127211,3,0.019795,0.195173,3,0.033450,0.270184,3,0.076357,0.646638,3,0.268344,0.260185,3,0.0,2.55102,3,0.0,0.000000,3,0.0,0.000000,3,0.000000e+00,0.000000,3,0.000000e+00,NaN,0,NaN,NaN,0,NaN,0.000000,3,0.0,NaN,0,NaN,NaN,0,NaN
2,hipporag,3,49,49,49,3,43,0.270040,3,0.0,0.114260,3,0.0,0.116263,3,0.0,2.673469,3,0.0,0.265306,3,0.000000e+00,0.053488,3,0.0,0.719528,3,0.0,0.781964,3,1.359740e-16,0.749001,3,0.000000e+00,0.187307,3,0.061071,0.199143,3,0.057567,0.509802,3,0.224971,0.836895,3,0.130467,0.201577,3,0.0,2.55102,3,0.0,79.979592,3,0.0,158.102041,3,0.000000e+00,0.345458,3,6.798700e-17,0.009490,3,0.0,0.044388,3,0.0,0.647933,3,0.0,0.013366,3,2.124594e-18,0.041964,3,0.0
3,hypergraphrag,3,49,49,49,3,43,0.203279,3,0.0,0.300364,3,0.0,0.162739,3,0.0,2.673469,3,0.0,0.183673,3,0.000000e+00,0.060078,3,0.0,0.810532,3,0.0,0.801310,3,0.000000e+00,0.805147,3,0.000000e+00,0.408315,3,0.070872,0.471541,3,0.024847,0.648342,3,0.110926,0.895581,3,0.084593,0.286587,3,0.0,2.55102,3,0.0,0.000000,3,0.0,0.000000,3,0.000000e+00,0.000000,3,0.000000e+00,NaN,0,NaN,NaN,0,NaN,0.000000,3,0.0,NaN,0,NaN,NaN,0,NaN
4,llmbased,3,49,49,49,3,43,0.318264,3,0.0,0.149108,3,0.0,0.143759,3,0.0,2.673469,3,0.0,0.000000,3,0.000000e+00,0.000000,3,0.0,0.783663,3,0.0,0.779654,3,0.000000e+00,0.781073,3,0.000000e+00,0.178571,3,0.024765,0.273469,3,0.107002,0.352041,3,0.050806,0.708367,3,0.126135,0.145957,3,0.0,2.55102,3,0.0,9.224490,3,0.0,11.489796,3,0.000000e+00,0.


numeric: 7 rows x 57 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,llm_judge_count,answer_token_precision_mean,answer_token_precision_judge_count,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_judge_count,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_judge_count,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_judge_count,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_judge_count,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_judge_count,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_judge_count,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_judge_count,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_judge_count,bertscore_f1_std,llm_completeness_mean,llm_completeness_judge_count,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_judge_count,llm_faithfulness_std,llm_relevance_mean,llm_relevance_judge_count,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_judge_count,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_judge_count,nli_entailment_max_std,numeric_ground_truth_count_mean,numeric_ground_truth_count_judge_count,numeric_ground_truth_count_std,numeric_predicted_count_mean,numeric_predicted_count_judge_count,numeric_predicted_count_std,numeric_accuracy_mean,numeric_accuracy_judge_count,numeric_accuracy_std
0,fullcontext,3,16,16,16,3,0.660500,3,0.0,0.345912,3,6.798700e-17,0.408937,3,6.798700e-17,2.8125,3,0.0,0.8125,3,0.0,0.125,3,0.0,0.824401,3,0.0,0.831342,3,0.0,0.827059,3,0.0,0.752083,3,0.025259,0.747917,3,0.003608,0.991667,3,0.009547,0.982083,3,0.016121,0.286214,3,0.000000e+00,2.8125,3,0.0,679.666667,3,0.0,0.0625,3,0.0
1,grasp,3,16,16,16,3,0.190341,3,0.0,0.049062,3,0.000000e+00,0.055236,3,0.000000e+00,2.8125,3,0.0,0.0000,3,0.0,0.000,3,0.0,0.820224,3,0.0,0.793875,3,0.0,0.804871,3,0.0,0.145833,3,0.036084,0.187500,3,0.062500,0.360417,3,0.085315,0.657917,3,0.282457,0.465113,3,0.000000e+00,2.8125,3,0.0,0.600000,3,0.0,0.1250,3,0.0
2,hipporag,3,16,16,16,3,0.372024,3,0.0,0.040831,3,0.000000e+00,0.064290,3,0.000000e+00,2.8125,3,0.0,0.0000,3,0.0,0.000,3,0.0,0.718650,3,0.0,0.767990,3,0.0,0.742070,3,0.0,0.252083,3,0.003608,0.333333,3,0.095470,0.625000,3,0.249687,1.000000,3,0.000000,0.315796,3,0.000000e+00,2.8125,3,0.0,1.666667,3,0.0,0.2500,3,0.0
3,hypergraphrag,3,16,16,16,3,0.297760,3,0.0,0.538288,3,0.000000e+00,0.363903,3,6.798700e-17,2.8125,3,0.0,0.1250,3,0.0,0.125,3,0.0,0.842689,3,0.0,0.867849,3,0.0,0.854512,3,0.0,0.320833,3,0.007217,0.416667,3,0.130104,0.643750,3,0.178864,0.974375,3,0.022212,0.324410,3,0.000000e+00,2.8125,3,0.0,2.666667,3,0.0,0.2500,3,0.0
4,llmbased,3,16,16,16,3,0.566587,3,0.0,0.188141,3,0.000000e+00,0.267850,3,0.000000e+00,2.8125,3,0.0,0.0000,3,0.0,0.000,3,0.0,0.825663,3,0.0,0.792304,3,0.0,0.807957,3,0.0,0.031250,3,0.018750,0.056250,3,0.076801,0.120833,3,0.088682,0.839583,3,0.078146,0.204821,3,3.399350e-17,2.8125,3,0.0,1.000000,3,0.0,0.1250,3,0.0



answer_winrate: 7 rows x 12 columns


,method,judge_count,comparisons_min,comparisons_max,wins_mean,wins_std,losses_mean,losses_std,ties_mean,ties_std,winrate_mean,winrate_std
0,LWE,3,494,618,435.333333,49.217206,132.000000,25.238859,9.333333,4.618802,0.764099,0.015084
1,fullcontext,3,90,103,51.333333,7.505553,46.000000,3.605551,1.333333,0.577350,0.525656,0.045345
2,grasp,3,79,103,12.000000,6.557439,82.333333,9.018500,0.666667,0.577350,0.125640,0.057966
3,hipporag,3,82,103,12.000000,5.291503,81.333333,11.590226,2.666667,2.081666,0.138152,0.042180
4,hypergraphrag,3,83,103,31.000000,8.888194,61.000000,12.000000,4.333333,2.886751,0.346181,0.074256



pairwise_answer_winrate: 1854 rows x 20 columns


,evaluation_judge_id,evaluation_config_path,ground_truth_id,ground_truth_qtype,question,ground_truth_answer,method_a,method_b,method_a_key,method_b_key,answer_a,answer_b,winner,judge_rationale,winning_method,method_a_score,method_b_score,judge_id,judge,judge_error
0,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_0,"[""multi"", ""numeric""]","\nHow many ""experiment execution"" are there in...",The answer to the question is 15 unique execut...,fullcontext,LWE,fullcontext,ours,There are 13 experiment executions in the prov...,There are **15** experiment executions represe...,method_b,Method B matches the ground truth answer of 15...,LWE,0.0,1.0,gpt-5.4-mini,GPT-5.4 mini,NaN
1,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_1,"[""multi"", ""entity""]",\nIn what places do we utilize AI in this work...,\nThe ChatBS System utilizes AI for the follow...,fullcontext,LWE,fullcontext,ours,"```json { ""answer"": ""The workflow employs AI (...",AI is employed in several distinct stages of t...,method_a,Method A is closer to the ground truth: it cor...,fullcontext,1.0,0.0,gpt-5.4-mini,GPT-5.4 mini,NaN
2,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_10,"[""multi"", ""entity""]",\nwhat are the input parameters of the functio...,\n The input parameters to sparql query fun...,fullcontext,LWE,fullcontext,ours,The SPARQL query builder function receives the...,The question asks for the *input parameters* o...,method_a,The ground truth indicates a single input port...,fullcontext,1.0,0.0,gpt-5.4-mini,GPT-5.4 mini,NaN
3,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_100,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",fullcontext,LWE,fullcontext,ours,The execution produces a single output on the ...,The execution identified by **http://testwebsi...,method_b,Method B is more faithful to the ground truth ...,LWE,0.0,1.0,gpt-5.4-mini,GPT-5.4 mini,NaN
4,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_101,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",fullcontext,LWE,fullcontext,ours,The execution **id_20260420105727_300** produc...,The execution “http://testwebsite/testProgram#...,method_b,Method B matches the ground truth by stating t...,LWE,0.0,1.0,gpt-5.4-mini,GPT-5.4 mini,NaN


## Inspect Available Columns

Use this cell when choosing the placeholders below.

In [4]:
for name, df in summaries.items():
    print(f"\n{name}")
    for column in df.columns:
        print(f"  - {column}")


bool
  - run
  - judge_count
  - evaluated_examples
  - evaluated_examples_min
  - evaluated_examples_max
  - llm_judge_count
  - answer_token_precision_mean
  - answer_token_precision_judge_count
  - answer_token_precision_std
  - answer_token_recall_mean
  - answer_token_recall_judge_count
  - answer_token_recall_std
  - answer_token_f1_mean
  - answer_token_f1_judge_count
  - answer_token_f1_std
  - gt_entity_total_mean
  - gt_entity_total_judge_count
  - gt_entity_total_std
  - gt_entity_covered_mean
  - gt_entity_covered_judge_count
  - gt_entity_covered_std
  - gt_entity_coverage_mean
  - gt_entity_coverage_judge_count
  - gt_entity_coverage_std
  - bertscore_precision_mean
  - bertscore_precision_judge_count
  - bertscore_precision_std
  - bertscore_recall_mean
  - bertscore_recall_judge_count
  - bertscore_recall_std
  - bertscore_f1_mean
  - bertscore_f1_judge_count
  - bertscore_f1_std
  - llm_completeness_mean
  - llm_completeness_judge_count
  - llm_completeness_std
  - ll

## Editable Paper Table Columns

Edit these lists to decide what goes into each paper table. Keep `run` or `method` first if you want method names visible.

In [5]:
# These columns show ensemble means and between-judge sample standard deviations.
bool_columns = [
    "run",
    "judge_count",
    "llm_judge_count",
    "evaluated_examples",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
    "bool_accuracy_mean",
    "bool_accuracy_std",
]

entity_columns = [
    "run",
    "recall_examples",
    "judge_count",
    "llm_judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
    "entity_recall_final_mean",
    "entity_recall_final_std",
    "entity_precision_final_mean",
    "entity_precision_final_std",
    "entity_f1_final_mean",
    "entity_f1_final_std",
    "entity_recall_total_mean",
    "entity_recall_total_std",
    "entity_precision_total_mean",
    "entity_precision_total_std",
    "entity_f1_total_mean",
    "entity_f1_total_std",
]

numeric_columns = [
    "run",
    "judge_count",
    "llm_judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "numeric_accuracy_mean",
    "numeric_accuracy_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
]

answer_winrate_columns = [
    "method",
    "judge_count",
    "comparisons_min",
    "comparisons_max",
    "winrate_mean",
    "winrate_std",
]

In [6]:
# Optional display names. Add, remove, or rename as needed for the paper.
column_labels = {
    "run": "Method",
    "method": "Method",
    "judge_count": "Result Sources",
    "llm_judge_count": "LLM Judges",
    "recall_examples": "Recall N",
    "evaluated_examples": "N",
    "evaluated_examples_min": "N Min",
    "evaluated_examples_max": "N Max",
    "comparisons_min": "Comparisons Min",
    "comparisons_max": "Comparisons Max",
    "answer_token_f1_mean": "Answer Token F1",
    "answer_token_f1_std": "Answer Token F1 SD",
    "bertscore_f1_mean": "BERTScore F1",
    "bertscore_f1_std": "BERTScore F1 SD",
    "nli_entailment_max_mean": "NLI Entailment",
    "nli_entailment_max_std": "NLI Entailment SD",
    "bool_accuracy_mean": "Bool Accuracy",
    "bool_accuracy_std": "Bool Accuracy SD",
    "entity_recall_final_mean": "Entity Recall",
    "entity_recall_final_std": "Entity Recall SD",
    "entity_precision_final_mean": "Entity Precision",
    "entity_precision_final_std": "Entity Precision SD",
    "entity_f1_final_mean": "Entity F1",
    "entity_f1_final_std": "Entity F1 SD",
    "entity_recall_total_mean": "Entity Recall Total",
    "entity_recall_total_std": "Entity Recall Total SD",
    "entity_precision_total_mean": "Entity Precision Total",
    "entity_precision_total_std": "Entity Precision Total SD",
    "entity_f1_total_mean": "Entity F1 Total",
    "entity_f1_total_std": "Entity F1 Total SD",
    "numeric_accuracy_mean": "Numeric Accuracy",
    "numeric_accuracy_std": "Numeric Accuracy SD",
    "llm_completeness_mean": "Completeness",
    "llm_completeness_std": "Completeness SD",
    "llm_completeness_judge_count": "Completeness Judges",
    "llm_relevance_mean": "Relevance",
    "llm_relevance_std": "Relevance SD",
    "llm_relevance_judge_count": "Relevance Judges",
    "llm_faithfulness_mean": "Faithfulness",
    "llm_faithfulness_std": "Faithfulness SD",
    "llm_faithfulness_judge_count": "Faithfulness Judges",
    "llm_understanderbility_mean": "Understandability",
    "llm_understanderbility_std": "Understandability SD",
    "llm_understanderbility_judge_count": "Understandability Judges",
    "winrate_mean": "Win Rate",
    "winrate_std": "Win Rate SD",
}

method_order = ["GWB", "VSB", "GRASP", "HippoRAG", "HyperGRAG", "Ours"]

method_labels = {
    "fullcontext": "FCB",
    "grasp": "GRASP",
    "hipporag": "HippoRAG",
    "hypergraphrag": "HyperGRAG",
    "llmbased": "GWB",
    "LWE": "Ours",
    "ours": "Ours",
    "vectorsimilarity": "VSB",
}

def order_method_rows(table, method_column="Method"):
    ordered = table[table[method_column].isin(method_order)].copy()
    ordered[method_column] = pd.Categorical(
        ordered[method_column],
        categories=method_order,
        ordered=True,
    )
    sort_columns = [method_column]
    if "Question Type" in ordered.columns:
        sort_columns = ["Question Type", method_column]
    return ordered.sort_values(sort_columns).reset_index(drop=True)

count_columns = {"Result Sources", "LLM Judges", "N", "N Min", "N Max", "Recall N", "Comparisons Min", "Comparisons Max"}

def scale_result_columns(table):
    scaled = table.copy()
    for position, column in enumerate(scaled.columns):
        if column in count_columns or column.endswith(" Judges") or column.endswith("_judge_count"):
            continue
        values = scaled.iloc[:, position]
        if pd.api.types.is_numeric_dtype(values):
            scaled.iloc[:, position] = values * 100
    return scaled

## Build Individual Tables

In [7]:
round_digits = 3

bool_table = summaries["bool"][bool_columns].copy()
bool_table["run"] = bool_table["run"].replace(method_labels)
bool_table = scale_result_columns(bool_table.rename(columns=column_labels)).round(round_digits)
bool_table = order_method_rows(bool_table)

entity_table = summaries["entity"][entity_columns].copy()
entity_table["run"] = entity_table["run"].replace(method_labels)
entity_table = scale_result_columns(entity_table.rename(columns=column_labels)).round(round_digits)
entity_table = order_method_rows(entity_table)

numeric_table = summaries["numeric"][numeric_columns].copy()
numeric_table["run"] = numeric_table["run"].replace(method_labels)
numeric_table = scale_result_columns(numeric_table.rename(columns=column_labels)).round(round_digits)
numeric_table = order_method_rows(numeric_table)



In [8]:
if "answer_winrate" in summaries:
    answer_winrate_table = summaries["answer_winrate"][answer_winrate_columns].copy()
    answer_winrate_table["method"] = answer_winrate_table["method"].replace(method_labels)
    answer_winrate_table = scale_result_columns(answer_winrate_table.rename(columns=column_labels)).round(round_digits)
    answer_winrate_table = order_method_rows(answer_winrate_table)
else:
    answer_winrate_table = pd.DataFrame(
        columns=[column_labels.get(column, column) for column in answer_winrate_columns]
    )

In [9]:
display(bool_table)
display(entity_table)
display(numeric_table)
display(answer_winrate_table)

,Method,Result Sources,LLM Judges,N,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD,Bool Accuracy,Bool Accuracy SD
0,GWB,3,3,38,77.851,0.0,27.776,0.0,0.000,0.0
1,VSB,3,3,38,79.052,0.0,47.551,0.0,34.211,0.0
2,GRASP,3,3,38,80.011,0.0,49.270,0.0,28.947,0.0
3,HippoRAG,3,3,38,73.511,0.0,26.060,0.0,52.632,0.0
4,HyperGRAG,3,3,38,81.902,0.0,50.359,0.0,42.105,0.0
5,Ours,3,3,38,83.548,0.0,48.683,0.0,71.053,0.0


,Method,Recall N,Result Sources,LLM Judges,N,Answer Token F1,Answer Token F1 SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD,Entity Recall,Entity Recall SD,Entity Precision,Entity Precision SD,Entity F1,Entity F1 SD,Entity Recall Total,Entity Recall Total SD,Entity Precision Total,Entity Precision Total SD,Entity F1 Total,Entity F1 Total SD
0,GWB,43,3,3,49,14.376,0.0,78.107,0.0,14.596,0.0,29.897,0.0,4.529,0.0,16.355,0.0,56.835,0.0,8.553,0.0,19.674,0.0
1,VSB,43,3,3,49,19.474,0.0,78.803,0.0,23.268,0.0,10.594,0.0,1.868,0.0,15.837,0.0,18.928,0.0,2.499,0.0,13.393,0.0
2,GRASP,43,3,3,49,9.741,0.0,80.548,0.0,26.018,0.0,0.000,0.0,NaN,NaN,NaN,NaN,0.000,0.0,NaN,NaN,NaN,NaN
3,HippoRAG,43,3,3,49,11.626,0.0,74.900,0.0,20.158,0.0,34.546,0.0,0.949,0.0,4.439,0.0,64.793,0.0,1.337,0.0,4.196,0.0
4,HyperGRAG,43,3,3,49,16.274,0.0,80.515,0.0,28.659,0.0,0.000,0.0,NaN,NaN,NaN,NaN,0.000,0.0,NaN,NaN,NaN,NaN
5,Ours,43,3,3,49,22.130,0.0,80.482,0.0,28.580,0.0,38.372,0.0,1.726,0.0,4.097,0.0,49.419,0.0,1.368,0.0,5.248,0.0


,Method,Result Sources,LLM Judges,N,Answer Token F1,Answer Token F1 SD,Numeric Accuracy,Numeric Accuracy SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD
0,GWB,3,3,16,26.785,0.0,12.50,0.0,80.796,0.0,20.482,0.0
1,VSB,3,3,16,24.688,0.0,6.25,0.0,79.533,0.0,43.944,0.0
2,GRASP,3,3,16,5.524,0.0,12.50,0.0,80.487,0.0,46.511,0.0
3,HippoRAG,3,3,16,6.429,0.0,25.00,0.0,74.207,0.0,31.580,0.0
4,HyperGRAG,3,3,16,36.390,0.0,25.00,0.0,85.451,0.0,32.441,0.0
5,Ours,3,3,16,50.375,0.0,43.75,0.0,86.751,0.0,48.304,0.0


,Method,Result Sources,Comparisons Min,Comparisons Max,Win Rate,Win Rate SD
0,GWB,3,78,103,7.223,7.415
1,VSB,3,82,103,19.287,1.710
2,GRASP,3,79,103,12.564,5.797
3,HippoRAG,3,82,103,13.815,4.218
4,HyperGRAG,3,83,103,34.618,7.426
5,Ours,3,494,618,76.410,1.508


## Combined Table Placeholder

This creates one compact table across question types. Edit `combined_columns` to choose the shared metrics.

In [10]:
combined_columns = [
    "task",
    "run",
    "judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
]

combined_parts = []
for task_name in ["bool", "entity", "numeric"]:
    part = summaries[task_name].copy()
    part.insert(0, "task", task_name)
    combined_parts.append(part)

combined_table = pd.concat(combined_parts, ignore_index=True)
combined_table = combined_table[combined_columns].copy()
combined_table["run"] = combined_table["run"].replace(method_labels)
combined_table = scale_result_columns(combined_table.rename(columns={**column_labels, "task": "Question Type"})).round(round_digits)
combined_table = order_method_rows(combined_table)

display(combined_table)

,Question Type,Method,Result Sources,N,Answer Token F1,Answer Token F1 SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD
0,bool,GWB,3,38,20.500,0.0,77.851,0.0,27.776,0.0
1,bool,VSB,3,38,28.414,0.0,79.052,0.0,47.551,0.0
2,bool,GRASP,3,38,8.512,0.0,80.011,0.0,49.270,0.0
3,bool,HippoRAG,3,38,0.000,0.0,73.511,0.0,26.060,0.0
4,bool,HyperGRAG,3,38,25.889,0.0,81.902,0.0,50.359,0.0
5,bool,Ours,3,38,32.231,0.0,83.548,0.0,48.683,0.0
6,entity,GWB,3,49,14.376,0.0,78.107,0.0,14.596,0.0
7,entity,VSB,3,49,19.474,0.0,78.803,0.0,23.268,0.0
8,entity,GRASP,3,49,9.741,0.0,80.548,0.0,26.018,0.0
9,entity,HippoRAG,3,49,11.626,0.0,74.900,0.0,20.158,0.0


## Overall Summary Table

LLM metrics use valid item-level scores across all question types within each judge. Recall uses entity questions only. Mean and sample SD are calculated across available judge means. Result Sources counts result sets, LLM Judges counts judges with quality scores, and each attribute also has its own valid judge count. A requested judge with no scores is missing, not a zero score.


In [11]:
overall_metric_names = [
    "answer_token_f1", "bertscore_f1", "nli_entailment_max",
    *LLM_METRICS, "entity_recall_final", "entity_recall_total",
]
overall_parts = []
for judge_id in JUDGE_IDS:
    for category in CATEGORY_NAMES:
        frame = read_csv_or_empty(judge_analysis_dir(judge_id) / category / "results.csv")
        if not frame.empty:
            overall_parts.append(frame.assign(judge_id=judge_id, question_type=category))
overall_source = pd.concat(overall_parts, ignore_index=True)
if overall_source.duplicated(["judge_id", "run", "ground_truth_id", "question_type"]).any():
    raise ValueError("Duplicate per-question scores in overall summary")
overall_rows = []
for (judge_id, run_name), group in overall_source.groupby(["judge_id", "run"]):
    row = {"judge_id": judge_id, "run": run_name, "evaluated_examples": len(group)}
    for metric in overall_metric_names:
        metric_rows = group[group["question_type"].eq("entity")] if metric.startswith("entity_recall_") else group
        values = pd.to_numeric(
            metric_rows.get(metric, pd.Series(np.nan, index=metric_rows.index)),
            errors="coerce",
        )
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_count"] = int(values.notna().sum())
    row["recall_examples"] = row["entity_recall_final_count"]
    overall_rows.append(row)
overall_judge_source = pd.DataFrame(overall_rows)
overall_summary_table = aggregate_judge_values(
    overall_judge_source, "run", [f"{metric}_mean" for metric in overall_metric_names],
)
overall_summary_table["run"] = overall_summary_table["run"].replace(method_labels)
overall_summary_table = scale_result_columns(overall_summary_table.rename(columns=column_labels)).round(round_digits)
overall_summary_table = order_method_rows(overall_summary_table)
display(overall_summary_table)
if overall_summary_table["LLM Judges"].lt(len(JUDGE_IDS)).any():
    print("Incomplete LLM ensemble: means and SD use available ratings. See the per-attribute judge counts and availability tables.")


,Method,Result Sources,N,N Min,N Max,LLM Judges,Recall N,Answer Token F1,answer_token_f1_judge_count,Answer Token F1 SD,BERTScore F1,bertscore_f1_judge_count,BERTScore F1 SD,NLI Entailment,nli_entailment_max_judge_count,NLI Entailment SD,Completeness,Completeness Judges,Completeness SD,Relevance,Relevance Judges,Relevance SD,Faithfulness,Faithfulness Judges,Faithfulness SD,Understandability,Understandability Judges,Understandability SD,Entity Recall,entity_recall_final_judge_count,Entity Recall SD,Entity Recall Total,entity_recall_total_judge_count,Entity Recall Total SD
0,GWB,3,103,103,103,3,43,18.563,3,0.0,78.430,3,0.0,20.373,3,0.0,12.696,3,0.653,26.067,3,5.682,18.379,3,5.270,73.216,3,10.868,29.897,3,0.0,56.835,3,0.0
1,VSB,3,103,103,103,3,43,23.582,3,0.0,79.008,3,0.0,35.439,3,0.0,21.540,3,0.606,59.161,3,17.678,27.420,3,5.206,94.106,3,4.299,10.594,3,0.0,18.928,3,0.0
2,GRASP,3,103,103,103,3,43,8.633,3,0.0,80.345,3,0.0,36.879,3,0.0,18.049,3,3.517,36.197,3,6.955,22.481,3,1.823,67.341,3,24.283,0.000,3,0.0,0.000,3,0.0
3,HippoRAG,3,103,103,103,3,43,6.530,3,0.0,74.280,3,0.0,24.110,3,0.0,27.232,3,3.717,69.089,3,11.266,31.280,3,2.266,92.083,3,6.428,34.546,3,0.0,64.793,3,0.0
4,HyperGRAG,3,103,103,103,3,43,22.946,3,0.0,81.806,3,0.0,37.337,3,0.0,38.678,3,3.745,61.227,3,7.617,48.196,3,6.384,93.545,3,5.353,0.000,3,0.0,0.000,3,0.0
5,Ours,3,103,103,103,3,43,30.244,3,0.0,82.587,3,0.0,39.060,3,0.0,69.239,3,5.148,91.968,3,8.675,65.443,3,8.486,89.269,3,10.071,38.372,3,0.0,49.419,3,0.0


## Export Tables

Exports CSV and LaTeX versions into `paper_tables/`. Comment out any table you do not need.

In [12]:
tables = {
    "bool_summary": bool_table,
    "entity_summary": entity_table,
    "numeric_summary": numeric_table,
    "answer_winrate_summary": answer_winrate_table,
    "combined_summary": combined_table,
    "overall_summary": overall_summary_table,
}

for table_name, table in tables.items():
    csv_path = TABLE_OUTPUT_DIR / f"{table_name}.csv"
    tex_path = TABLE_OUTPUT_DIR / f"{table_name}.tex"

    table.to_csv(csv_path, index=False)
    table.to_latex(tex_path, index=False, escape=False)

    print(f"Wrote {csv_path}")
    print(f"Wrote {tex_path}")


availability.to_csv(TABLE_OUTPUT_DIR / "judge_artifact_availability.csv", index=False)
for category, source in judge_sources.items():
    source.to_csv(TABLE_OUTPUT_DIR / f"{category}_summary_by_judge.csv", index=False)
overall_judge_source.to_csv(TABLE_OUTPUT_DIR / "overall_summary_by_judge.csv", index=False)
if not winrate_source.empty:
    winrate_source.to_csv(TABLE_OUTPUT_DIR / "answer_winrate_summary_by_judge.csv", index=False)
if "pairwise_answer_winrate" in summaries:
    summaries["pairwise_answer_winrate"].to_csv(TABLE_OUTPUT_DIR / "pairwise_answer_winrate_by_judge.csv", index=False)


Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/bool_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/bool_summary.tex


Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/entity_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/entity_summary.tex


Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/numeric_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/numeric_summary.tex
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/answer_winrate_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/answer_winrate_summary.tex


Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/combined_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/combined_summary.tex
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/overall_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/overall_summary.tex


## Optional: Pairwise Win Rate Details

In [13]:
if "pairwise_answer_winrate" in summaries:
    pairwise_table = summaries["pairwise_answer_winrate"].copy()
    for column in pairwise_table.columns:
        if pairwise_table[column].dtype == "object":
            pairwise_table[column] = pairwise_table[column].replace(method_labels)
    pairwise_table = scale_result_columns(pairwise_table).round(round_digits)
    display(pairwise_table)

,evaluation_judge_id,evaluation_config_path,ground_truth_id,ground_truth_qtype,question,ground_truth_answer,method_a,method_b,method_a_key,method_b_key,answer_a,answer_b,winner,judge_rationale,winning_method,method_a_score,method_b_score,judge_id,judge,judge_error
0,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_0,"[""multi"", ""numeric""]","\nHow many ""experiment execution"" are there in...",The answer to the question is 15 unique execut...,fullcontext,LWE,fullcontext,ours,There are 13 experiment executions in the prov...,There are **15** experiment executions represe...,method_b,Method B matches the ground truth answer of 15...,LWE,0.0,100.0,gpt-5.4-mini,GPT-5.4 mini,NaN
1,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_1,"[""multi"", ""entity""]",\nIn what places do we utilize AI in this work...,\nThe ChatBS System utilizes AI for the follow...,fullcontext,LWE,fullcontext,ours,"```json { ""answer"": ""The workflow employs AI (...",AI is employed in several distinct stages of t...,method_a,Method A is closer to the ground truth: it cor...,fullcontext,100.0,0.0,gpt-5.4-mini,GPT-5.4 mini,NaN
2,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_10,"[""multi"", ""entity""]",\nwhat are the input parameters of the functio...,\n The input parameters to sparql query fun...,fullcontext,LWE,fullcontext,ours,The SPARQL query builder function receives the...,The question asks for the *input parameters* o...,method_a,The ground truth indicates a single input port...,fullcontext,100.0,0.0,gpt-5.4-mini,GPT-5.4 mini,NaN
3,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_100,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",fullcontext,LWE,fullcontext,ours,The execution produces a single output on the ...,The execution identified by **http://testwebsi...,method_b,Method B is more faithful to the ground truth ...,LWE,0.0,100.0,gpt-5.4-mini,GPT-5.4 mini,NaN
4,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_101,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",fullcontext,LWE,fullcontext,ours,The execution **id_20260420105727_300** produc...,The execution “http://testwebsite/testProgram#...,method_b,Method B matches the ground truth by stating t...,LWE,0.0,100.0,gpt-5.4-mini,GPT-5.4 mini,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1849,gemma-4-31b,/home/desild/work/research/LLM-Workflow-Explor...,gt_95,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",grasp,LWE,grasp,ours,NaN,The execution `http://testwebsite/testProgram#...,method_b,"Method A provided no answer, whereas Method B ...",LWE,0.0,100.0,gemma-4-31b,Gemma 4 31B,NaN
1850,gemma-4-31b,/home/desild/work/research/LLM-Workflow-Explor...,gt_96,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",grasp,LWE,grasp,ours,<|channel|>final <|constrain|>answer<|message|...,The execution identified by `http://testwebsit...,method_b,Method B correctly identifies the number of ou...,LWE,0.0,100.0,gemma-4-31b,Gemma 4 31B,NaN
1851,gemma-4-31b,/home/desild/work/research/LLM-Workflow-Explor...,gt_97,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",grasp,LWE,grasp,ours,NaN,The execution identified by `http://testwebsit...,method_b,Method B correctly identifies the number of ou...,LWE,0.0,100.0,gemma-4-31b,Gemma 4 31B,NaN
1852,gemma-4-31b,/home/desild/work/research/LLM-Workflow-Explor...,gt_98,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",grasp,LWE,grasp,ours,NaN,The execution `http://testwebsite/testProg

## Krippendorff Alpha Agreement

This final section measures how consistently the LLM judges assign the **completeness**, **relevance**, and **faithfulness** scores used in the paper tables. It computes **interval Krippendorff's alpha** directly from the item-level `[0, 1]` ratings in each judge's raw `results.csv`; squared-distance disagreement is appropriate because the attributes are numeric rather than unordered labels.

A rating item is aligned by question type, method, and ground-truth ID. The overall rows cover the same six methods retained in the paper tables, and the remaining rows show agreement for each method separately. Items with at least two available ratings are usable, so an absent, failed, or partially completed judge run does not prevent calculation. Alpha is `1` for perfect agreement, approximately `0` when agreement is no better than expected from the pooled ratings, and can be negative for systematic disagreement. This item-level reliability measure complements the between-judge standard deviations of the aggregate means above.

Only GPT, Qwen, and Gemma are selected. Judges without valid ratings do not contribute to alpha; the output distinguishes usable Judges from Requested Judges. Llama is excluded.


In [14]:
AGREEMENT_ATTRIBUTES = {
    "llm_completeness": "Completeness",
    "llm_relevance": "Relevance",
    "llm_faithfulness": "Faithfulness",
}
AGREEMENT_METHODS = [
    "llmbased",
    "vectorsimilarity",
    "grasp",
    "hipporag",
    "hypergraphrag",
    "ours",
]
ITEM_COLUMNS = ["question_type", "run", "ground_truth_id"]


def load_item_level_judge_ratings():
    frames = []
    rows = []
    for judge_id in JUDGE_IDS:
        for question_type in CATEGORY_NAMES:
            path = judge_analysis_dir(judge_id) / question_type / "results.csv"
            frame = read_csv_or_empty(path)
            available_attributes = [column for column in AGREEMENT_ATTRIBUTES if column in frame.columns]
            usable = (
                not frame.empty and {"run", "ground_truth_id"}.issubset(frame.columns)
                and bool(available_attributes)
                and frame[available_attributes].apply(pd.to_numeric, errors="coerce").notna().any().any()
            )
            rows.append({
                "judge_id": judge_id,
                "judge": JUDGE_LABELS[judge_id],
                "question_type": question_type,
                "available": usable,
                "rows": len(frame),
                "attributes": ", ".join(available_attributes),
                "path": str(path.relative_to(REPO_ROOT)),
            })
            if not usable or not available_attributes:
                continue
            selected = frame[["run", "ground_truth_id", *available_attributes]].copy()
            selected.insert(0, "question_type", question_type)
            selected.insert(0, "judge", JUDGE_LABELS[judge_id])
            selected.insert(0, "judge_id", judge_id)
            frames.append(selected)
    source = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return source, pd.DataFrame(rows)


def krippendorff_alpha_interval(ratings):
    numeric = ratings.apply(pd.to_numeric, errors="coerce")
    usable = numeric.loc[numeric.notna().sum(axis=1).ge(2)]
    if usable.empty:
        return None
    observed_numerator = 0.0
    pooled_ratings = []
    for _, item_ratings in usable.iterrows():
        values = item_ratings.dropna().to_numpy(dtype=float)
        rating_count = len(values)
        centered_sum_squares = float(((values - values.mean()) ** 2).sum())
        observed_numerator += 2 * rating_count * centered_sum_squares / (rating_count - 1)
        pooled_ratings.extend(values)
    pooled = np.asarray(pooled_ratings, dtype=float)
    ratings_per_item = usable.notna().sum(axis=1)
    total_ratings = len(pooled)
    observed_disagreement = observed_numerator / total_ratings
    expected_disagreement = 2 * float(((pooled - pooled.mean()) ** 2).sum()) / (total_ratings - 1)
    alpha = 1 - observed_disagreement / expected_disagreement if expected_disagreement > 0 else np.nan
    return {
        "items_total": len(numeric),
        "items_used": len(usable),
        "ratings_used": total_ratings,
        "min_ratings_per_item": int(ratings_per_item.min()),
        "max_ratings_per_item": int(ratings_per_item.max()),
        "observed_disagreement": observed_disagreement,
        "expected_disagreement": expected_disagreement,
        "krippendorff_alpha": alpha,
    }


agreement_source, agreement_availability = load_item_level_judge_ratings()
alpha_rows = []
if not agreement_source.empty:
    agreement_source["ground_truth_id"] = agreement_source["ground_truth_id"].astype(str)
    agreement_source = agreement_source[agreement_source["run"].isin(AGREEMENT_METHODS)].copy()
    if agreement_source.duplicated(["judge_id", *ITEM_COLUMNS]).any():
        raise ValueError("Duplicate attribute ratings found for the same judge and item")

    for attribute, attribute_label in AGREEMENT_ATTRIBUTES.items():
        if attribute not in agreement_source.columns:
            continue
        rating_matrix = agreement_source.pivot(
            index=ITEM_COLUMNS,
            columns="judge_id",
            values=attribute,
        )
        scopes = [("overall", "all", "All paper methods", rating_matrix)]
        scopes.extend(
            (
                "method",
                method,
                method_labels.get(method, method),
                rating_matrix[rating_matrix.index.get_level_values("run") == method],
            )
            for method in AGREEMENT_METHODS
        )
        for scope_type, scope, scope_label, scoped_matrix in scopes:
            scope_judges = [
                judge_id for judge_id in JUDGE_IDS
                if judge_id in scoped_matrix.columns and scoped_matrix[judge_id].notna().any()
            ]
            stats = krippendorff_alpha_interval(scoped_matrix[scope_judges])
            if stats is not None:
                alpha_rows.append({
                    "attribute": attribute_label,
                    "scope_type": scope_type,
                    "scope": scope,
                    "scope_label": scope_label,
                    "distance": "interval_squared",
                    "judge_count": len(scope_judges),
                    "requested_judge_count": len(JUDGE_IDS),
                    "judges": ", ".join(scope_judges),
                    **stats,
                })

krippendorff_agreement = pd.DataFrame(alpha_rows)
alpha_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_agreement.csv"
availability_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_input_availability.csv"
krippendorff_agreement.to_csv(alpha_path, index=False)
agreement_availability.to_csv(availability_path, index=False)
print(f"Wrote {alpha_path}")
print(f"Wrote {availability_path}")
if krippendorff_agreement.empty:
    print("At least two judges with overlapping item-level attribute ratings are required for alpha.")
else:
    alpha_overall_table = krippendorff_agreement.loc[
        krippendorff_agreement["scope_type"].eq("overall"),
        ["attribute", "judge_count", "requested_judge_count", "items_used", "ratings_used", "krippendorff_alpha"],
    ].rename(columns={
        "attribute": "Attribute",
        "judge_count": "Judges",
        "requested_judge_count": "Requested Judges",
        "items_used": "Items",
        "ratings_used": "Ratings",
        "krippendorff_alpha": "Krippendorff Alpha",
    })
    overall_csv_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_overall.csv"
    overall_tex_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_overall.tex"
    alpha_overall_table.round(3).to_csv(overall_csv_path, index=False)
    alpha_overall_table.round(3).to_latex(overall_tex_path, index=False, escape=False)
    print(f"Wrote {overall_csv_path}")
    print(f"Wrote {overall_tex_path}")
    display(alpha_overall_table.round(3))
    display(krippendorff_agreement.loc[krippendorff_agreement["scope_type"].eq("method")].round(3))

Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/krippendorff_alpha_agreement.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/krippendorff_alpha_input_availability.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/krippendorff_alpha_overall.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/krippendorff_alpha_overall.tex


,Attribute,Judges,Requested Judges,Items,Ratings,Krippendorff Alpha
0,Completeness,3,3,618,1846,0.856
7,Relevance,3,3,618,1846,0.672
14,Faithfulness,3,3,618,1846,0.656


,attribute,scope_type,scope,scope_label,distance,judge_count,requested_judge_count,judges,items_total,items_used,ratings_used,min_ratings_per_item,max_ratings_per_item,observed_disagreement,expected_disagreement,krippendorff_alpha
1,Completeness,method,llmbased,GWB,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",103,103,308,2,3,0.026,0.106,0.757
2,Completeness,method,vectorsimilarity,VSB,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",103,103,308,2,3,0.047,0.257,0.816
3,Completeness,method,grasp,GRASP,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",103,103,308,2,3,0.059,0.256,0.771
4,Completeness,method,hipporag,HippoRAG,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",103,103,305,2,3,0.053,0.310,0.830
5,Completeness,method,hypergraphrag,HyperGRAG,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",103,103,308,2,3,0.055,0.408,0.866
6,Completeness,method,ours,Ours,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",103,103,309,3,3,0.064,0.344,0.814
8,Relevance,method,llmbased,GWB,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",103,103,308,2,3,0.053,0.139,0.618
9,Relevance,method,vectorsimilarity,VSB,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",103,103,308,2,3,0.245,0.347,0.294
10,Relevance,method,grasp,GRASP,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",103,103,308,2,3,0.102,0.416,0.754
11,Relevance,method,hipporag,HippoRAG,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",103,103,305,2,3,0.186,0.337,0.448
